# Hyperparameter Tuning

## What is Hyperparameter Tuning?

Hyperparameter tuning automatically searches for the best combination of hyperparameters to maximize model performance. SageMaker uses Bayesian optimization to intelligently explore the parameter space, reducing the number of training jobs needed.

## Creating a HyperparameterTuner

In [ ]:
from sagemaker.tuner import HyperparameterTuner, IntegerParameter, ContinuousParameter, CategoricalParameter
from sagemaker.xgboost import XGBoost
import sagemaker

session = sagemaker.Session()
role = 'arn:aws:iam::123456789012:role/SageMakerRole'
bucket = session.default_bucket()

# Create base estimator
xgb_estimator = XGBoost(
    entry_point='train.py',
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    framework_version='1.5',
    output_path=f's3://{bucket}/xgb-output',
    sagemaker_session=session
)

# Define hyperparameter ranges
hyperparameter_ranges = {
    'max_depth': IntegerParameter(1, 10),
    'eta': ContinuousParameter(0.1, 0.5),
    'min_child_weight': IntegerParameter(2, 10),
    'subsample': ContinuousParameter(0.5, 1.0),
    'gamma': ContinuousParameter(0, 5)
}

# Create tuner
tuner = HyperparameterTuner(
    estimator=xgb_estimator,
    objective_metric_name='validation:auc',
    hyperparameter_ranges=hyperparameter_ranges,
    metric_definitions=[
        {'Name': 'validation:auc', 'Regex': 'validation-auc=([0-9\\.]+)'}
    ],
    max_jobs=20,
    max_parallel_jobs=4,
    base_tuning_job_name='xgb-tuning'
)

# Start tuning
tuner.fit(
    {'training': f's3://{bucket}/train-data/'},
    job_name='xgb-tuning-job'
)

## Bayesian Optimization Strategy

In [ ]:
from sagemaker.tuner import HyperparameterTuner, StrategyConfig

# Configure Bayesian optimization
strategy_config = StrategyConfig(
    strategy='Bayesian',
    metric_definitions=[
        {'Name': 'validation:accuracy', 'Regex': 'accuracy=([0-9\\.]+)'}
    ]
)

tuner = HyperparameterTuner(
    estimator=xgb_estimator,
    objective_metric_name='validation:accuracy',
    hyperparameter_ranges=hyperparameter_ranges,
    max_jobs=30,
    max_parallel_jobs=5,
    strategy='Bayesian'
)

# Bayesian optimization learns from previous trials
tuner.fit({'training': f's3://{bucket}/train-data/'})

## Warm Start for Tuning

In [ ]:
from sagemaker.tuner import WarmStartConfig, WarmStartTypes

# Use results from previous tuning job
warm_start_config = WarmStartConfig(
    type=WarmStartTypes.TRANSFER_LEARNING,
    from_job_name='xgb-tuning-job-2024-01-10'
)

# Create new tuner with warm start
tuner = HyperparameterTuner(
    estimator=xgb_estimator,
    objective_metric_name='validation:auc',
    hyperparameter_ranges=hyperparameter_ranges,
    max_jobs=20,
    max_parallel_jobs=4,
    warm_start_config=warm_start_config
)

tuner.fit({'training': f's3://{bucket}/train-data/'})

## Tuning Job Configuration

```json
{
  "tuning_job_config": {
    "tuning_job_name": "xgb-tuning-job",
    "tuning_objective": {
      "metric_name": "validation:auc",
      "type": "Maximize"
    },
    "resource_limits": {
      "max_number_of_training_jobs": 20,
      "max_parallel_training_jobs": 4
    },
    "parameter_ranges": {
      "integer_parameter_ranges": [
        {
          "name": "max_depth",
          "min_value": "1",
          "max_value": "10"
        }
      ],
      "continuous_parameter_ranges": [
        {
          "name": "eta",
          "min_value": "0.1",
          "max_value": "0.5"
        }
      ]
    },
    "strategy_config": {
      "strategy": "Bayesian"
    }
  }
}
```

## Analyzing Tuning Results

In [ ]:
# Get best training job
best_job = tuner.best_training_job()
print(f"Best job: {best_job}")

# Get best hyperparameters
best_hyperparameters = tuner.best_estimator().hyperparameters()
print(f"Best hyperparameters: {best_hyperparameters}")

# Deploy best model
best_predictor = tuner.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large'
)

## Early Stopping for Efficiency

In [ ]:
from sagemaker.tuner import HyperparameterTuner

tuner = HyperparameterTuner(
    estimator=xgb_estimator,
    objective_metric_name='validation:auc',
    hyperparameter_ranges=hyperparameter_ranges,
    max_jobs=20,
    max_parallel_jobs=4,
    early_stopping_type='Auto'  # Enable automatic early stopping
)

tuner.fit({'training': f's3://{bucket}/train-data/'})

## Quiz 1

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is the primary purpose of hyperparameter tuning?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="0">
      <span>To train models faster</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="1">
      <span>To find optimal hyperparameters automatically</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="2">
      <span>To reduce data size</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="3">
      <span>To deploy models</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 2

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What optimization strategy does SageMaker use by default?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="0">
      <span>Bayesian optimization</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="1">
      <span>Grid search</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="2">
      <span>Random search</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="3">
      <span>Manual tuning</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 3

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ What is warm start in hyperparameter tuning?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="0">
      <span>Starting training with warm data</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="1">
      <span>Pre-warming instances</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="2">
      <span>Using results from previous tuning jobs</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="3">
      <span>Warming up the model</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 4

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What does early stopping do in tuning?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="0">
      <span>Stops all training jobs</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="1">
      <span>Stops underperforming jobs to save time and cost</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="2">
      <span>Stops the tuning job early</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="3">
      <span>Stops data loading</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 5

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What is the objective metric in tuning?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="0">
      <span>The metric to optimize (maximize or minimize)</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="1">
      <span>The training time</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="2">
      <span>The cost of tuning</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="3">
      <span>The number of jobs</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>